
# Notebook 1


In [0]:
%sql
SELECT * FROM workspace.default.wine_quality LIMIT 10

## Define catalog & schema space

In [0]:
%sql
USE CATALOG 'fashion_trends_based_on_fashion_products_and_outfit_combinations';
-- USE SCHEMA  'default';
SELECT current_catalog(), current_schema();
SELECT * FROM fashion_products LIMIT 100

In [0]:
%sql
USE CATALOG 'workspace';
DESCRIBE SCHEMA EXTENDED default; --describe the schema metadata
SHOW tables in default; -- display the table in the current schema (under this catalog)
SHOW volumes in default

#### To query a volume file (say csv), you can use below code (note that xlsx file is not supported, due to multiple tabs it supports)

In [0]:
spark.sql(f'''
          SELECT *
          FROM text.'/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv' -- This will fail as we do not have volume files. But it will give you a preview of csv file
          ''').display()

In [0]:
%sql
-- Another way to to read a csv volumne file in SQL directly (assuming your csv first row is column name)
SELECT *
FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv',
  format => 'csv',
  header => true,
  inferSchema => true -- 
)

### Create the delta table from a volume csv file

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.user_query;

-- Create the delta table using csv 
CREATE TABLE workspace.default.user_query AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
    
-- Display the table
SELECT * FROM workspace.default.user_query

### In python, you can also read the volume csv file directly and create a spark dataframe


In [0]:
# read the csv file aand create the spark dataframe
sdf = spark.read.format('csv').option('header','true').option('inferschema','true').load('/Volumes/workspace/default/test_volume/test_directory/synthetic_user_query_data.csv')

# create the delta table from the spark dataframe
sdf.write.format('delta').mode('overwrite').saveAsTable('workspace.default.user_query')


In [0]:
# Read the created delta table using python
spark.read.table('workspace.default.user_query').display()